In [1]:
import pandas as pd
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_cause_and_scenario

In [2]:
location = "india"
vehicle = "rice"

In [3]:
# Parameters
location = "india"
vehicle = "rice"


In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylls = pd.read_parquet(path)
else:
    pregnancy_ylls = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylls.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_ylls

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,ylls,cause,other_causes,other_causes,10_to_14,invalid,1,baseline,0,2,0.0
1,ylls,cause,other_causes,other_causes,10_to_14,invalid,2,baseline,0,2,0.0
2,ylls,cause,other_causes,other_causes,10_to_14,invalid,3,baseline,0,2,0.0
3,ylls,cause,other_causes,other_causes,10_to_14,invalid,4,baseline,0,2,0.0
4,ylls,cause,other_causes,other_causes,10_to_14,invalid,5,baseline,0,2,0.0
...,...,...,...,...,...,...,...,...,...,...,...
26995,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,1,baseline,0,9,0.0
26996,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,2,baseline,0,9,0.0
26997,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,3,baseline,0,9,0.0
26998,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,4,baseline,0,9,0.0


In [6]:
pregnancy_ylls.groupby("scenario").random_seed.nunique()

scenario
baseline        10
intervention    10
zero            10
Name: random_seed, dtype: int64

In [7]:
assert (pregnancy_ylls[pregnancy_ylls.value > 0].entity == "maternal_disorders").all()

In [8]:
pregnancy_ylls_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylls).pipe(
    lambda df: df[df.index.get_level_values("entity") == "maternal_disorders"]
)
pregnancy_ylls_by_scenario

scenario      entity              wealth_quintile
baseline      maternal_disorders  1                  560242.077730
                                  2                  358880.906231
                                  3                  405179.925721
                                  4                  341317.808230
                                  5                  211480.114359
intervention  maternal_disorders  1                  560242.077730
                                  2                  358880.906231
                                  3                  405179.925721
                                  4                  341317.808230
                                  5                  211480.114359
zero          maternal_disorders  1                  563290.855460
                                  2                  368754.727354
                                  3                  413748.474075
                                  4                  341317.808230
            

In [9]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylds = pd.read_parquet(path)
else:
    pregnancy_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

pregnancy_ylds

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,ylds,cause,pregnancy,pregnant,10_to_14,invalid,1,baseline,0,2,0.0
1,ylds,cause,pregnancy,parturition,10_to_14,invalid,1,baseline,0,2,0.0
2,ylds,cause,pregnancy,postpartum,10_to_14,invalid,1,baseline,0,2,0.0
3,ylds,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,1,baseline,0,2,0.0
4,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,10_to_14,invalid,1,baseline,0,2,0.0
...,...,...,...,...,...,...,...,...,...,...,...
94495,ylds,cause,pregnancy,postpartum,95_plus,severe,5,baseline,0,9,0.0
94496,ylds,cause,maternal_disorders,maternal_disorders,95_plus,severe,5,baseline,0,9,0.0
94497,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,95_plus,severe,5,baseline,0,9,0.0
94498,ylds,cause,all_causes,all_causes,95_plus,severe,5,baseline,0,9,0.0


In [10]:
# Pregnancy has no disability, and maternal hemorrhage disability is counted in maternal_disorders
assert (
    pregnancy_ylds[
        pregnancy_ylds.entity.isin(["pregnancy", "maternal_hemorrhage"])
    ].value
    == 0
).all()

In [11]:
pregnancy_ylds_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylds).pipe(
    lambda df: df[
        ~df.index.get_level_values("entity").isin(["pregnancy", "maternal_hemorrhage"])
    ]
)
pregnancy_ylds_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                   96821.283068
                                  2                   76036.968770
                                  3                   60816.045385
                                  4                   52350.680881
                                  5                   40806.063973
              maternal_disorders  1                     376.083832
                                  2                     202.958717
                                  3                     222.006288
                                  4                     216.310848
                                  5                     144.441052
intervention  anemia              1                   96821.283068
                                  2                   76036.968770
                                  3                   60816.045385
                                  4                   52350.680881
            

In [12]:
pregnancy_dalys_by_scenario = pregnancy_ylls_by_scenario.add(
    pregnancy_ylds_by_scenario, fill_value=0
)
pregnancy_dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                   96821.283068
                                  2                   76036.968770
                                  3                   60816.045385
                                  4                   52350.680881
                                  5                   40806.063973
              maternal_disorders  1                  560618.161562
                                  2                  359083.864948
                                  3                  405401.932008
                                  4                  341534.119077
                                  5                  211624.555411
intervention  anemia              1                   96821.283068
                                  2                   76036.968770
                                  3                   60816.045385
                                  4                   52350.680881
            

In [13]:
ylds_path = f"results/rescaled_child_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(ylds_path).is_file():
    # The child model has no disease models enabled, so it must contribute no YLDs.
    # The current results system writes explicit zero-valued rows where older
    # versions wrote an empty table, so check the values rather than emptiness.
    assert (pd.read_parquet(ylds_path).value == 0).all()

In [14]:
path = f"results/rescaled_child_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    neonatal_ylls = pd.read_parquet(path).rename(
        columns={"maternal_scenario": "scenario"}
    )
else:
    neonatal_ylls = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/ylls.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_ylls

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,input_draw,random_seed,value
0,ylls,cause,other_causes,other_causes,0_to_5_months,Female,1,baseline,intervention,0,2,640778.872031
1,ylls,cause,other_causes,other_causes,0_to_5_months,Female,2,baseline,intervention,0,2,489265.923297
2,ylls,cause,other_causes,other_causes,0_to_5_months,Female,3,baseline,intervention,0,2,484799.776437
3,ylls,cause,other_causes,other_causes,0_to_5_months,Female,4,baseline,intervention,0,2,450231.535335
4,ylls,cause,other_causes,other_causes,0_to_5_months,Female,5,baseline,intervention,0,2,467580.095052
...,...,...,...,...,...,...,...,...,...,...,...,...
1195,ylls,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,0,6,79527.549117
1196,ylls,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,0,6,62943.046988
1197,ylls,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,0,6,54762.885237
1198,ylls,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,0,6,33571.282938


In [15]:
neonatal_ylls_by_scenario = aggregate_by_cause_and_scenario(neonatal_ylls)
assert (
    neonatal_ylls_by_scenario[
        neonatal_ylls_by_scenario.index.get_level_values("entity") != "other_causes"
    ]
    == 0
).all()
neonatal_ylls_by_scenario = neonatal_ylls_by_scenario[
    neonatal_ylls_by_scenario.index.get_level_values("entity") == "other_causes"
]
neonatal_ylls_by_scenario = (
    neonatal_ylls_by_scenario.reset_index()
    .assign(entity="lbwsg")
    .set_index(neonatal_ylls_by_scenario.index.names)
    .value
)
neonatal_ylls_by_scenario

scenario      entity  wealth_quintile
baseline      lbwsg   1                  1.716716e+07
                      2                  1.429567e+07
                      3                  1.290847e+07
                      4                  1.174660e+07
                      5                  1.160855e+07
intervention  lbwsg   1                  1.716716e+07
                      2                  1.429567e+07
                      3                  1.290847e+07
                      4                  1.174660e+07
                      5                  1.160855e+07
zero          lbwsg   1                  1.720182e+07
                      2                  1.428701e+07
                      3                  1.295612e+07
                      4                  1.175093e+07
                      5                  1.162154e+07
Name: value, dtype: float64

In [16]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_ylds = pd.read_parquet(path)
else:
    non_pregnancy_anemia_ylds = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/ylds.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_ylds

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,1186.490411,zero
1,Female,0.0,0.019178,2,901.389895,zero
2,Female,0.0,0.019178,3,798.971870,zero
3,Female,0.0,0.019178,4,664.417092,zero
4,Female,0.0,0.019178,5,465.048259,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,80.978228,intervention
746,Male,95.0,125.000000,2,75.451506,intervention
747,Male,95.0,125.000000,3,77.357440,intervention
748,Male,95.0,125.000000,4,72.631383,intervention


In [17]:
# For comparison with previous round of results, we also look at
# WRA and U5
wra_non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[
        (non_pregnancy_anemia_ylds.sex == "Female")
        & (non_pregnancy_anemia_ylds.age_start >= 10)
        & (non_pregnancy_anemia_ylds.age_end <= 55)
    ].assign(entity="anemia", input_draw="draw_0")
)
wra_non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  1.810846e+06
                      2                  1.772233e+06
                      3                  1.816683e+06
                      4                  1.734604e+06
                      5                  1.580374e+06
intervention  anemia  1                  1.810846e+06
                      2                  1.772233e+06
                      3                  1.816683e+06
                      4                  1.734604e+06
                      5                  1.580374e+06
zero          anemia  1                  1.935722e+06
                      2                  1.894235e+06
                      3                  1.926864e+06
                      4                  1.828468e+06
                      5                  1.629746e+06
Name: value, dtype: float64

In [18]:
scenarios[1]

'zero'

In [19]:
(
    wra_non_pregnancy_anemia_ylds_by_scenario.loc["baseline"].sum()
    + pregnancy_ylds_by_scenario.loc[("baseline", "anemia")].sum()
) - (
    wra_non_pregnancy_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
    + pregnancy_ylds_by_scenario.loc[(scenarios[1], "anemia")].sum()
)

-528479.5606238563

In [20]:
u5_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds[(non_pregnancy_anemia_ylds.age_end <= 5)].assign(
        entity="anemia", input_draw="draw_0"
    )
)
u5_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  622350.110058
                      2                  507914.038386
                      3                  468649.860085
                      4                  402170.554909
                      5                  314830.860156
intervention  anemia  1                  622350.110058
                      2                  507914.038386
                      3                  468649.860085
                      4                  402170.554909
                      5                  314830.860156
zero          anemia  1                  657232.983445
                      2                  537238.145707
                      3                  492948.098560
                      4                  420532.322873
                      5                  323175.185458
Name: value, dtype: float64

In [21]:
(
    u5_anemia_ylds_by_scenario.loc["baseline"].sum()
    - u5_anemia_ylds_by_scenario.loc[scenarios[1]].sum()
)

-115211.31245009555

In [22]:
non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(
    non_pregnancy_anemia_ylds.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  1                  4.360566e+06
                      2                  3.959792e+06
                      3                  3.915742e+06
                      4                  3.614767e+06
                      5                  3.222801e+06
intervention  anemia  1                  4.360566e+06
                      2                  3.959792e+06
                      3                  3.915742e+06
                      4                  3.614767e+06
                      5                  3.222801e+06
zero          anemia  1                  4.664061e+06
                      2                  4.232154e+06
                      3                  4.153325e+06
                      4                  3.809427e+06
                      5                  3.321418e+06
Name: value, dtype: float64

In [23]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ylls_by_scenario.csv"
if pathlib.Path(path).is_file():
    neural_tube_defect_ylls_by_scenario = pd.read_csv(path)
else:
    neural_tube_defect_ylls_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/intervention/ylls_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

neural_tube_defect_ylls_by_scenario = neural_tube_defect_ylls_by_scenario.set_index(
    ["scenario", "entity", "wealth_quintile"]
).value
neural_tube_defect_ylls_by_scenario

scenario      entity  wealth_quintile
zero          ntd     1                  363252.670950
                      2                  320895.824642
                      3                  288089.031881
                      4                  271653.759488
                      5                  231356.103631
baseline      ntd     1                  337619.988843
                      2                  301581.649030
                      3                  273425.007964
                      4                  259634.600477
                      5                  226772.618242
intervention  ntd     1                  191469.436097
                      2                  184057.388590
                      3                  178650.637949
                      4                  178416.881760
                      5                  189908.314111
Name: value, dtype: float64

In [24]:
dalys_by_scenario = (
    pregnancy_dalys_by_scenario.add(neonatal_ylls_by_scenario, fill_value=0)
    .add(non_pregnancy_anemia_ylds_by_scenario, fill_value=0)
    .add(neural_tube_defect_ylls_by_scenario, fill_value=0)
)
dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              1                  4.457387e+06
                                  2                  4.035829e+06
                                  3                  3.976558e+06
                                  4                  3.667118e+06
                                  5                  3.263608e+06
              lbwsg               1                  1.716716e+07
                                  2                  1.429567e+07
                                  3                  1.290847e+07
                                  4                  1.174660e+07
                                  5                  1.160855e+07
              maternal_disorders  1                  5.606182e+05
                                  2                  3.590839e+05
                                  3                  4.054019e+05
                                  4                  3.415341e+05
                          

In [25]:
import pathlib

In [26]:
path = f"./results/{location}/{vehicle}/dalys_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
dalys_by_scenario.to_csv(path)